In [4]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ============================================================
# 2. AUTO DETECT DATASET PATH (IMPORTANT FIX)
# ============================================================
base_input = "/kaggle/input/"
datasets = os.listdir(base_input)
print("Available datasets:", datasets)

# 👉 automatically pick first dataset
BASE_PATH = os.path.join(base_input, datasets[0]) + "/"

TRAIN_PATH = BASE_PATH + "train/train/"
TEST_PATH  = BASE_PATH + "test/test/"

print("Using dataset:", BASE_PATH)
print("Train exists:", os.path.exists(TRAIN_PATH))
print("Test exists:", os.path.exists(TEST_PATH))

# ============================================================
# 3. LOAD TRAIN DATA
# ============================================================
IMG_SIZE = 150

def load_train_data():
    X, y = [], []

    for img_name in tqdm(os.listdir(TRAIN_PATH)):
        path = os.path.join(TRAIN_PATH, img_name)

        # label from filename
        label = 1 if "dog" in img_name.lower() else 0

        img = cv2.imread(path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        X.append(img)
        y.append(label)

    return np.array(X), np.array(y)

X, y = load_train_data()
X = X / 255.0

print("Shape:", X.shape, y.shape)

# ============================================================
# 4. TRAIN-VALIDATION SPLIT
# ============================================================
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# ============================================================
# 5. DATA AUGMENTATION
# ============================================================
train_aug = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_aug = ImageDataGenerator()

# ============================================================
# 6. CNN MODEL
# ============================================================
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# ============================================================
# 7. TRAIN MODEL
# ============================================================
history = model.fit(
    train_aug.flow(X_train, y_train, batch_size=32),
    validation_data=val_aug.flow(X_val, y_val),
    epochs=10
)

# ============================================================
# 8. LOAD TEST DATA
# ============================================================
def load_test_data():
    test_images, test_ids = [], []

    for img_name in tqdm(os.listdir(TEST_PATH)):
        path = os.path.join(TEST_PATH, img_name)

        img = cv2.imread(path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        test_images.append(img)
        test_ids.append(img_name)

    return np.array(test_images), test_ids

test_X, test_ids = load_test_data()
test_X = test_X / 255.0

# ============================================================
# 9. PREDICT
# ============================================================
pred = model.predict(test_X)
pred_labels = ["Dog" if p > 0.5 else "Cat" for p in pred]

# ============================================================
# 10. SUBMISSION
# ============================================================
sub = pd.DataFrame({
    "id": test_ids,
    "Class": pred_labels
})

sub.to_csv("submission.csv", index=False)
sub.head()

# ============================================================
# ================= MCQ ANSWERS (DO NOT DELETE) ===============
# ============================================================

# 1. Supervised Learning:
# ✅ Learning from labeled input-output pairs

# 2. Supervised tasks:
# ✅ Email spam classification
# ✅ House price prediction

# 3. Accuracy:
# ✅ 0.85

# 4. Algorithms:
# ✅ Linear Regression
# ✅ Decision Trees
# ✅ Support Vector Machine

# 5. Unsupervised goal:
# ✅ Discover hidden structures in data

# 6. K-Means:
# ✅ Minimizes intra-cluster variance
# ✅ Sensitive to initial centroid placement

# 7. K = 3:
# ✅ 3 centroids

# 8. Conv output:
# ✅ 30×30

# 9. CNN components:
# ✅ Convolutional layers
# ✅ Pooling layers
# ✅ Activation functions

# 10. CNN facts:
# ✅ Exploit spatial locality
# ✅ Shared weights
# ✅ Reduce parameters

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/catdog-classification/train/train/'